> **English archival edition.** Saved outputs are historical observations, preserved rather than rerun. Chinese text has been translated. Use a new run label or output directory: translated source hashes differ from historical checkpoints.

# v7.2 · 8 scenarios protocol diagnosis: allow normal tasks to be completed first 

** Complete standalone document: 8 A new scenarios, 576 Real tool tracks, task and code are all built up, no need for old notebook Or research the head. **

 This round of positioning. v7.1 . No list is available. / Before History / After history, and Base Original text, Instruct Original List user Sealed, sealed. Instruct Separation system Command three. The original model is modified, BF16、 Retain three tools, greedy decoding, and six free calls. There are no head interventions or OpenAI API cost. 

 Its... ** Small-scale technical diagnosis **， Number of independent samples 8；576 is the number of paired conditional trajectories. Do not present this round as new mechanistic confirmation. All ordinary failures, format errors, and no orders remain in the denominator. 

##  How? 

1.  Use with v7.1 Same.  **A100 80GB**。 in Colab Secrets Settings  `HF_TOKEN`， Open the book. notebook Access rights, two accounts. Llama Access permission for models. 
2.  Default  `RUN_LABEL="main"`、`RUN_MODE="all"`， Click  **Run all**。 Restart if first install runtime Later. Run all。
3.  Default 6 hour soft budget; budget first completes the current trajectory and saves PARTIAL and synchronize Drive Released after release. GPU。 The same label can be renewed. Not yet measured GPU Time. 6 Hours are not a guarantee. 
4.  Check the real order after all is done. ZIP CRC , and file summaries, save to Drive， then disconnect automatically. The webpage need not remain open, but continued background execution still depends on Colab Session. 

 Results catalogue: `MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_v7p2_protocol_diag8/protocol-main/`。 The old experimental catalogue will not be changed. 


In [ ]:
#@title  Set with  Drive  Mount 
import time, os, re, math
from pathlib import Path
from google.colab import drive
V72_STARTED=time.monotonic()
RUN_LABEL="main" #@param {type:"string"}
RUN_MODE="all" #@param ["all", "export"]
MAX_WALL_HOURS=6.0 #@param {type:"number"}
AUTO_DISCONNECT=True #@param {type:"boolean"}
TAKEOVER_LOCK_ID="" #@param {type:"string"}
V72_STUDY=V72_RESULT=V72_SESSION=None
if RUN_MODE not in ("all","export") or not re.fullmatch(r"[A-Za-z0-9_-]{1,48}",RUN_LABEL):
    raise ValueError("RUN_MODE or RUN_LABEL Illegal. ")
if not isinstance(MAX_WALL_HOURS,(int,float)) or not math.isfinite(MAX_WALL_HOURS) or not 0<MAX_WALL_HOURS<=23:
    raise ValueError("MAX_WALL_HOURS to be greater than 0 and not more than 23 Hours. ")
DRIVE_MOUNT=Path("/content/drive")
drive.mount(str(DRIVE_MOUNT))
if not os.path.ismount(str(DRIVE_MOUNT)) or not (DRIVE_MOUNT/"MyDrive").is_dir():
    raise RuntimeError("Drive Mounting failed. Please complete the mount before running. ")
V72_BASE=DRIVE_MOUNT/"MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_v7p2_protocol_diag8"
V72_RUN=V72_BASE/("protocol-"+RUN_LABEL)
print(" Results catalogue: ",V72_RUN,flush=True)


In [ ]:
if RUN_MODE != "export":
    import sys, subprocess, importlib.metadata as v5_metadata
    if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
        raise RuntimeError(' Fixed-dependent combination needs  Python 3.10–3.13。')
    V6_REQUIRED_VERSIONS = {
        'transformer-lens': '2.17.0', 'transformers': '4.57.1', 'huggingface-hub': '0.36.0',
        'numpy': '2.1.3' if sys.version_info[:2] == (3, 13) else '1.26.4',
        'pandas': '2.2.3', 'plotly': '6.1.2', 'scipy': '1.15.3',
        'scikit-learn': '1.6.1', 'datasets': '3.6.0', 'pyarrow': '20.0.0',
    }
    V6_INSTALL_PROBLEMS = []
    for v5_name, v5_version in V6_REQUIRED_VERSIONS.items():
        try:
            v5_installed = v5_metadata.version(v5_name)
        except v5_metadata.PackageNotFoundError:
            v5_installed = None
        if v5_installed != v5_version:
            V6_INSTALL_PROBLEMS.append((v5_name, v5_installed, v5_version))
    if V6_INSTALL_PROBLEMS:
        print(' The original experimental fixed version will be installed: ', V6_INSTALL_PROBLEMS)
        subprocess.run([sys.executable, '-m', 'pip', 'install',
                        *[f'{name}=={version}' for name, version in V6_REQUIRED_VERSIONS.items()], 'tqdm'], check=True)
        raise SystemExit(' Installation complete. Please restart  Colab runtime， Again.  Run all； Versions are not repeated after matching. ')
    print(' Fixed reliance matches, skips installation. ')

else:
    print(" Read-only export: Skip this step. ")


In [ ]:
if RUN_MODE != "export":
    #  Run this block after restarting: check if the installation is currently in place  Python  Environment. 
    import sys
    import importlib.metadata as metadata
    import importlib.util

    print("Python:", sys.version.split()[0])
    print(" Interpreter :", sys.executable)
    if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
        raise RuntimeError(" Yes.  Python 3.10–3.13； Please re-implement the installment block after running. ")

    expected_versions = {
        "transformer-lens": "2.17.0", "transformers": "4.57.1",
        "huggingface-hub": "0.36.0",
        "numpy": "2.1.3" if sys.version_info[:2] == (3, 13) else "1.26.4",
        "pandas": "2.2.3",
        "plotly": "6.1.2", "scipy": "1.15.3", "scikit-learn": "1.6.1",
        "datasets": "3.6.0", "pyarrow": "20.0.0",
    }
    problems = []
    for package, expected in expected_versions.items():
        try:
            installed = metadata.version(package)
        except metadata.PackageNotFoundError:
            installed = None
        print(f"{package}: {installed if installed is not None else ' not installed '} ( Yes.  {expected})")
        if installed != expected:
            problems.append(f"{package}:  current  {installed}， Yes.  {expected}")

    for package in ("torch", "tqdm"):
        try:
            print(f"{package}: {metadata.version(package)}")
        except metadata.PackageNotFoundError:
            problems.append(f"{package}:  not installed ")

    try:
        torch_version = metadata.version("torch")
        torch_major_minor = tuple(int(part) for part in torch_version.split("+")[0].split(".")[:2])
        if torch_major_minor < (2, 6):
            problems.append(f"torch:  current  {torch_version}， Yes.  >=2.6")
    except metadata.PackageNotFoundError:
        pass  #  The record is missing. 
    except ValueError:
        problems.append(" Cannot parse  torch  Version, please check the current version  Colab  Run when. ")

    if importlib.util.find_spec("transformer_lens") is None:
        problems.append(" current  Python  Cannot find  transformer_lens  module ")
    if problems:
        raise RuntimeError(
            " Reliance check failed: \n- " + "\n- ".join(problems)
            + "\n Run the first cell and inspect its complete installation log. After successful installation, restart the runtime, "
              " Go on from this block. "
        )
    print(" Reliance on a version check pass; continues to run the third code block. ")

else:
    print(" Read-only export: Skip this step. ")


In [ ]:
# Authenticate before accessing gated Hugging Face model files.
from google.colab import userdata
from huggingface_hub import login
try:
    _archive_hf_token = userdata.get("HF_TOKEN")
except Exception:
    raise RuntimeError("Set HF_TOKEN in Colab Secrets and enable notebook access.") from None
if not isinstance(_archive_hf_token, str) or not _archive_hf_token.strip():
    raise RuntimeError("HF_TOKEN is empty. Set it in Colab Secrets and enable notebook access.")
login(token=_archive_hf_token.strip(), add_to_git_credential=False)
del _archive_hf_token


In [ ]:
if RUN_MODE != "export":
    from google.colab import userdata
    from huggingface_hub import login
    def authenticate_hub():
        hint = " Please.  Colab Secrets  Set Readable  Llama  Model.  HF_TOKEN， ♪ And open this book ♪  notebook  Access rights. "
        try:
            token = userdata.get("HF_TOKEN")
        except Exception:
            raise RuntimeError(hint) from None
        if not isinstance(token,str) or not token.strip():
            raise RuntimeError(hint)
        try:
            login(token=token.strip(), add_to_git_credential=False)
        except Exception:
            raise RuntimeError(hint) from None
    authenticate_hub()

else:
    print(" Read-only export: Skip this step. ")


In [ ]:
#@title  Loading Run Library ( export Model only, standard library, not loaded GPU model) 
import base64,hashlib,json,zlib,importlib.metadata
V72_DEPS={}
V72_PACKAGES={}
if RUN_MODE!="export":
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
    import numpy as np
    import torch
    from transformer_lens import HookedTransformer
    from huggingface_hub import get_token
    def disabled_interactive_login72():
        raise RuntimeError("HF login invalid; setup Colab Secrets HF_TOKEN and rerun the login grid. ")
    V72_DEPS=dict(np=np,torch=torch,HookedTransformer=HookedTransformer,
                  get_token=get_token,notebook_login=disabled_interactive_login72)
    V72_NAMES=("torch","transformer-lens","transformers","huggingface-hub","numpy","pandas",
               "datasets","pyarrow","scipy","scikit-learn","plotly")
    V72_PACKAGES={k:importlib.metadata.version(k) for k in V72_NAMES}
    print("GPU：",torch.cuda.get_device_name(0) if torch.cuda.is_available() else " Unconnected ")


In [ ]:
#@title  Unpackage and verify frozen source, 8 A new scenarios and 576 Queue task (no model called) 
V72_ASSETS={name:base64.b64decode(value) for name,value in json.loads(zlib.decompress(base64.b64decode('...'))).items()}
V72_EXPECTED_DIGESTS={'code/bootstrap72.py': '5f9f73944427803dfe781a8a95f2efd84e23eb8ac9358fd4f4e6fbdc04b67a0b', 'code/tasks72.py': 'cfb8268a3b162af3a03c825cff9992bd54416df88c894aec5ed6bc56c659cd68', 'code/reports72.py': '3c65659a1dd96e7d4201c6ac293aa856046628c410bf7c5515ba71b7766230c6', 'code/study72.py': 'ab02438badeebf30bfa4f37a450db5eff4e8e405dfbac5d29881f2315a161f2d', 'code/runtime72.py': '26e7640eb133018eed45ba64cd93d91acd75d38ad4395127f78408087f462375', 'code/frozen/source_digests.json': '0a9af3e8d8ff061168dedc1a1483e9e206749932203f5602975f65ac1c37face', 'code/frozen/v61_aligned_source.py': '39fec76288d585eeb7ed01f35b61bfe8f2f51d3bff24f8fddbaa0266ff1c31bb', 'code/frozen/v6_core_source.py': 'dfee0014eb1746e959d1b652d152093f00ea90782932900bd2123ca5afee7c53', 'code/frozen/v6_legacy_source.py': 'ecc77fb8592acfda16f19342d276672567fff0c65fec292f92abec4d4b5bf1af', 'code/frozen/v6_runtime_source.py': '02759ddf3a5c525a0f22d558ff2b4aedf25f1afc3fa09987a3f8bc2ec51b1bb7', 'code/frozen/v6_tasks_source.py': '4e9f82772209d475ea35d57a9dc6c4044281a63bc07c47a574848972e0212f59', 'code/frozen/v70_engine70_source.py': 'a9bd44bcfdac1e3fd5e4efe90cc72bd479c7d000033467dffa52bb364cee1bbc', 'code/frozen/v70_tasks70_source.py': 'f26b83c06043f2b3c4d08d601aac9b7fd01cce6a4212d57965b9e50083e236cf', 'code/frozen/v71_model_identity.json': 'ba52ac8f52b96d3a5c082289d1bc8fa1fbefbfa7735ad495ce9be842eb021c01', 'inputs/protocol.md': '923da9e904cf628eb5c410885e37e150a0ed130b15c0ab63350b4cc0b05c1e7c', 'inputs/worlds.json': '7a98996e4a281775985ce71c5e27ca9a7b5f277abe355629fb0bf115c642019d', 'inputs/cells.json': '8e110688cce7556a06c27aae00d6a5a21eeb21fb19e26e122329761e83c279d2', 'inputs/design.json': '3e21432035a2551605cbd8a536cb8bcce60698b77f48751fe7b9f9985d86d17d', 'inputs/forbidden_identifiers.json': '18420ea413218c6b62f1f3e3be961cdbe4c8513c130f45c0345697763651f759', 'inputs/collision-audit.json': 'ac2e3daec993bc4b7e3d8439f7a3bf17f65d8c8ee199b0360b10660ae9cab02a', 'inputs/planned-grid.csv': 'fe84ab5d94d5da3253db1671b2b20d904015d58db00e61a2cc2fced39fd64fb3', 'inputs/example_prompt.txt': '0b26a7eb9f1d27fa7517829b7059a7ecc9a2cc27c733a579329b8fc287c31d7a', 'inputs/example_role_split.json': 'cfcd50c00f59ecc544d411e2cc4bb7855034177f9606ec5e9c636487b6f16556'}
if {k:hashlib.sha256(v).hexdigest() for k,v in V72_ASSETS.items()}!=V72_EXPECTED_DIGESTS:
    raise RuntimeError(" Embedded source code or input summary changes. ")
V72_BOOT={}
exec(compile(V72_ASSETS["code/bootstrap72.py"].decode(),"bootstrap72.py","exec"),V72_BOOT)
V72_NS=V72_BOOT["load_namespace72"](V72_ASSETS,V72_DEPS)
V72_WORLDS=json.loads(V72_ASSETS["inputs/worlds.json"])
V72_CELLS=json.loads(V72_ASSETS["inputs/cells.json"])
V72_NS["audit_worlds72"](V72_WORLDS,json.loads(V72_ASSETS["inputs/forbidden_identifiers.json"]))
print(V72_NS["audit_grid72"](V72_WORLDS,V72_CELLS))


##  How do you judge this round? 

 Every protocol. / There are conditions to the directory. 32 Normal trajectories (empty history) 16 Well check it out after double check. 16 Article ). Pre-fixed threshold for subsequent availability: at least 30/32 Strictly successful. At least two starting points. 15/16 Success, no more branches with formatting or tools failed 1 trajectories. Generate recommendations only; do not remove conditions mid-run or automatically launch a large experiment. 

 Separately inspect conditional recovery after one wrong lookup and after two lookups ending with the wrong item. Outputs X/Y/ Stand-by. / Other / Well go back and check it out. / incorrect reservations, order attempts, repeated queries, and budget exhaustion. Fewer incorrect purchases without an order do not constitute improvement. First aggregate each paired contrast with equal weights within scenarios, 8 The scenarios. bootstrap The area is described only. 

Instruct split Only put the same instructions and examples system， Current Task & Original JSON Tool History user； No replacement to original function calling。 Before directory / Only the same text block will be moved and the inventory will remain unchanged from the true history. 


In [ ]:
#@title  Start or resume: two models, all 576 Tracks 
V72_RESULT=V72_SESSION=None
with V72_NS["writer72"](V72_RUN,TAKEOVER_LOCK_ID):
    V72_IDENTITY=V72_NS["identity72"](V72_RUN,V72_ASSETS,V72_PACKAGES,export=RUN_MODE=="export")
    V72_STUDY=V72_NS["Study72"](V72_RUN,V72_ASSETS,V72_IDENTITY,export=RUN_MODE=="export")
    try:
        V72_SESSION=V72_NS["run72"](V72_STUDY,RUN_MODE,V72_STARTED,MAX_WALL_HOURS)
    except (Exception,KeyboardInterrupt) as error:
        V72_SESSION={"status":"ERROR","error_type":type(error).__name__}
        print(" Implementation interrupted. Completed tracks are still in place. Drive。 Runs the next cell to export PARTIAL； A continuation with the tag will skip the verified module. ",flush=True)
        raise
print(" Status: ",V72_SESSION["status"])


In [ ]:
#@title  Reload actual orders, recalculate reports and export ZIP（ No Model Call) 
if V72_STUDY is None:
    raise RuntimeError(" Please run the upper grid first; only export settings RUN_MODE='export' after Run all。")
with V72_NS["writer72"](V72_RUN):
    V72_RESULT=V72_NS["export72"](V72_STUDY,V72_SESSION)
print(" Outcome State: ",V72_RESULT["report"]["status"])
print(" Completed: ",V72_RESULT["report"]["completed_trajectories"],"/ 576")
print(" Replay Real Tools /ZIP Verify: ",V72_RESULT["validation"])
print("Drive ZIP：",V72_RESULT["path"])
for row in V72_RESULT["report"]["eligibility"]:
    print(row["protocol"],row["layout"]," Normal Success ",str(row["strict_success"])+"/"+str(row["n"]),row["status"])


##  Relay and File 

-  Interrupt or 6 Keep the same after the hour budget stops. RUN_LABEL Restart Run all； Completed trajectories JSON Skipped after signature and tool replay check. JSON Damage would be clearly stopped and could not be removed to cover up the error. 
-  If the hint locks, confirm the old one first. runtime Its stopped, and it is accurate. ID Fill to TAKEOVER_LOCK_ID。 Do not allow two runtime Write the same label. 
-  Export only: `RUN_MODE="export"`， Available CPU， Skip installation, login and model loading. ZIP The actual trajectory is still re-reconciled. 
- COMPLETE Yes. 576 The article is all done. `reports/report.json` Medium scope as `v7p2_protocol_diag8`。PARTIAL、 Folders and progress.json None indicates completion. 
-  Raw outputs, rendered prompts, tool calls, and final database states for every step are in `experiment/branches/`； The report sheet is... `reports/`； Freezing of data / Agreement / The exact source code is contained in ZIP Here we go. 
-  directory none good performance is a basic task-competence anchor, not a substitute for conditions requiring catalog M The successor agreement does not automatically trigger a new experiment. 


In [ ]:
#@title ZIP Verify sum Drive Auto-release after sync succeeded GPU
from google.colab import drive,runtime
V72_NS["finish72"](V72_STUDY,V72_RESULT,V72_SESSION,AUTO_DISCONNECT and RUN_MODE=="all",
    DRIVE_MOUNT,os.path.ismount,drive.flush_and_unmount,runtime.unassign)
